# 📚 LeetCode 滑动窗口 & 四数之和 & SQL 复习笔记

> **题目**：LC 209 · LC 18 · LC 176  
> **题型**：变长滑动窗口 · 多层循环 + 对撞双指针 · SQL（DISTINCT + LIMIT OFFSET + NULL处理）  
> **用途**：面试前快速回忆题型、触发条件、模板

---

## 目录

| # | 题号 | 题名 | 题型 | 难度 |
|---|------|------|------|------|
| 1 | 209 | 长度最小的子数组 | 变长滑动窗口 | 🟡 中等 |
| 2 | 18  | 四数之和 | 排序 + 双层循环 + 对撞双指针 | 🟡 中等 |
| 3 | 176 | 第二高的薪水 | SQL（DISTINCT + LIMIT OFFSET + NULL）| 🟢 简单/中等 |
| 4 | —   | 模式对照总结 | — | — |

In [ ]:
from typing import List
import pandas as pd

print("环境就绪 ✅")

---
<a id='lc209'></a>
## 1. LC 209 — 长度最小的子数组（变长滑动窗口）

| 字段 | 内容 |
|------|------|
| **题号** | 209 |
| **难度** | 🟡 中等 |
| **题型** | 变长滑动窗口（Variable-size Sliding Window）|
| **时间复杂度** | O(n) |
| **空间复杂度** | O(1) |
| **数据范围** | 1 ≤ n ≤ 10⁵，1 ≤ target ≤ 10⁹，1 ≤ nums[i] ≤ 10⁴（**全正数**）|

### 题目核心
找出和 **≥ target** 的**最短连续子数组**长度，不存在则返回 0。

### ⚡ 触发条件（何时用滑动窗口？）

| # | 信号 | 说明 |
|---|------|------|
| 1 | 求**连续子数组/子串**满足某条件 | 滑动窗口的本质对象 |
| 2 | 数组元素**全为正数**（或非负）| 保证"窗口右移总和增大、左移总和减小"的单调性 |
| 3 | 求满足条件的**最短 / 最长**长度 | "最短"→窗口满足后尽量收缩；"最长"→窗口不满足时收缩 |
| 4 | 暴力枚举所有子数组是 O(n²)，需要 O(n) | 双指针滑动，每个元素均摊 O(1) |

> ⚠️ **关键前提**：必须保证**单调性**（这里靠"全为正数"）。  
> 若数组含负数，`total` 不再单调，**右指针扫描时无法安全收缩左边界** —— 此时滑动窗口失效，需改用前缀和+哈希表（如 LC 1248）。

---

### 🧠 算法（"求最短"型滑动窗口）

```
lk = 0            # 左指针
total = 0         # 当前窗口和
min_len = inf     # 记录最小长度

for rk in range(n):              # 右指针主动扩张
    total += nums[rk]             # 扩窗：把新元素加入

    while total >= target:        # 窗口已满足条件 → 尝试收缩
        min_len = min(min_len, rk - lk + 1)
        total -= nums[lk]          # 缩窗：吐出左边元素
        lk += 1                     # 左指针右移

return min_len if min_len != inf else 0
```

**核心心法**：
- **rk（右指针）** 像"吸气"——不断把新元素纳入窗口，使窗口变大、变得更容易满足条件。
- **lk（左指针）** 像"呼气"——一旦满足条件，就疯狂收缩左边界（用 `while` 不是 `if`！），直到不再满足，从而找到**当前右端点下能取到的最短窗口**。
- `while` 而非 `if` 的原因：可能一次右移就让窗口"超额满足"很多，需要持续收缩到刚好不满足为止，否则会错过更短的解。

---

### "求最短" vs "求最长" 滑动窗口对比

| | 求最短（本题）| 求最长（如 LC 3 无重复最长子串）|
|--|--|--|
| 何时收缩左边界 | 条件**满足**时收缩 | 条件**不满足**时收缩 |
| 何时更新答案 | 收缩**之前**（条件满足的瞬间）| 每轮右移**之后**（窗口合法时）|
| 收缩用 while/if | `while`（尽量缩到最短）| `while`（缩到刚好合法）|
| 答案初值 | `inf`，取 `min` | `0`，取 `max` |

In [ ]:
# ===== 通用模板：变长滑动窗口（求最短）=====
# 触发：连续子数组 + 全正数（单调性）+ 求满足条件的最短长度
# 复杂度：O(n) time | O(1) space

def sliding_window_min_len(nums: List[int], target: int) -> int:
    n = len(nums)
    lk, total, min_len = 0, 0, float('inf')

    for rk in range(n):
        total += nums[rk]               # 扩窗：右指针主动前进

        while total >= target:          # 满足条件 → 尽量收缩
            min_len = min(min_len, rk - lk + 1)
            total -= nums[lk]            # 缩窗：吐出左边元素
            lk += 1

    return min_len if min_len != float('inf') else 0

print("✅ 滑动窗口（求最短）模板已加载")

In [ ]:
# ===== LC 209 完整解法 =====
# O(n) time | O(1) space
# lk = 左指针（窗口起点），rk = 右指针（窗口终点，主动扩张）

class Solution_209:
    def minSubArrayLen(self, target: int, nums: List[int]) -> int:
        n = len(nums)
        lk = 0
        total = 0
        min_len = float('inf')   # 先设一个无穷大，用来存最小长度

        # 让右指针 rk 主动向右扫描
        for rk in range(n):
            total += nums[rk]      # 右指针每走一步，就把新元素加进总和

            # 当总和 >= target 时，尝试压缩左边（缩小窗口）
            while total >= target:
                min_len = min(min_len, rk - lk + 1)   # 更新当前的最小长度
                total -= nums[lk]                      # 吐出左边的元素
                lk += 1                                 # 左指针右移

        # 如果 min_len 没变过，说明全加起来都不够，返回 0
        return min_len if min_len != float('inf') else 0

print("✅ LC 209 解法已加载")

In [ ]:
# ===== 测试用例 =====

sol209 = Solution_209()

tests209 = [
    (7,  [2, 3, 1, 2, 4, 3],         2),   # 示例1：[4,3]
    (4,  [1, 4, 4],                  1),   # 示例2：[4]
    (11, [1, 1, 1, 1, 1, 1, 1, 1],   0),   # 示例3：和最大8 < 11，无解
    (4,  [1, 1, 1, 1, 1, 1, 1, 1],   4),   # 刚好需要4个1
    (15, [1, 2, 3, 4, 5],            5),   # 全部加起来刚好15
    (3,  [1, 1, 1, 1, 1],            3),   # 连续3个1
]

all_pass = True
for target, nums, expected in tests209:
    result = sol209.minSubArrayLen(target, nums)
    ok = result == expected
    if not ok:
        all_pass = False
    flag = "✅" if ok else "❌"
    print(f"{flag}  target={target}, nums={nums}  →  {result}  (期望 {expected})")

print()
print("🎉 全部通过！" if all_pass else "⚠️ 有用例未通过")

---
<a id='lc18'></a>
## 2. LC 18 — 四数之和（排序 + 双层循环 + 对撞双指针）

| 字段 | 内容 |
|------|------|
| **题号** | 18 |
| **难度** | 🟡 中等 |
| **题型** | 排序 + 双层循环固定两数 + 对撞双指针 |
| **时间复杂度** | O(n³)：排序 O(n log n) + 三层循环 O(n³) |
| **空间复杂度** | O(1)（不计输出，排序原地）|
| **数据范围** | 1 ≤ n ≤ 200，−10⁹ ≤ nums[i], target ≤ 10⁹（**注意：可能溢出 int32，但 Python 不用担心**）|

### 题目核心
找出所有满足 `nums[a]+nums[b]+nums[c]+nums[d] == target` 且 `a,b,c,d` 互不相同、  
**不重复**的四元组，任意顺序返回。

### ⚡ 触发条件

| 信号 | 操作 |
|------|------|
| 找**固定数量 k 个元素**之和 = target，k ≥ 3 | 排序 + (k-2) 层循环固定 + 最内层对撞双指针 |
| 答案**不含重复**元组 | 每一层循环都要去重 |
| 数据范围小（n ≤ 200）| O(n^(k-1)) 可接受 |

---

### 🧠 kSum 通用降维思想

> **k 数之和 → 固定 (k-2) 个数 → 转化为两数之和（LC 167）用对撞双指针解决**

| 题目 | k | 外层循环数 | 内层 |
|------|---|-----------|------|
| LC 167 两数之和 II | 2 | 0 | 对撞双指针 |
| LC 15 三数之和 | 3 | 1 层（固定 i）| 对撞双指针 |
| **LC 18 四数之和** | 4 | **2 层（固定 i, j）** | 对撞双指针 |

LC 18 = 在 LC 15 外面再套一层循环，本质完全一致。

---

### 🧠 算法框架

```
nums.sort()

for i in range(n-3):
    if i > 0 and nums[i] == nums[i-1]: continue        # 去重①
    # 剪枝：当前最小和已经 > target，后面只会更大，直接结束
    if nums[i]+nums[i+1]+nums[i+2]+nums[i+3] > target: break
    # 剪枝：当前最大和仍 < target，i 这一层不可能满足，跳过
    if nums[i]+nums[n-3]+nums[n-2]+nums[n-1] < target: continue

    for j in range(i+1, n-2):
        if j > i+1 and nums[j] == nums[j-1]: continue   # 去重②
        if nums[i]+nums[j]+nums[j+1]+nums[j+2] > target: break
        if nums[i]+nums[j]+nums[n-2]+nums[n-1] < target: continue

        left, right = j+1, n-1
        while left < right:
            total = nums[i]+nums[j]+nums[left]+nums[right]
            if total == target:
                记录四元组
                去重③④（left, right，同 LC 15）
                left += 1; right -= 1
            elif total < target: left += 1
            else:                right -= 1
```

---

### 🔍 四层去重 + 双重剪枝详解

| 类型 | 代码 | 作用 |
|------|------|------|
| 去重① (i 层) | `if i>0 and nums[i]==nums[i-1]: continue` | 与 LC15 外层去重相同，向前比较 |
| 去重② (j 层) | `if j>i+1 and nums[j]==nums[j-1]: continue` | 同理，但条件是 `j > i+1`（j 的"前一个"必须也在合法范围内）|
| 去重③④ (left/right) | 同 LC 15 | 跳过双指针重复值 |
| **剪枝-下界 (break)** | 当前最小可能和 > target → 直接终止本层循环 | `nums[i]` 已排序，更大的 i/j 只会让和更大 |
| **剪枝-上界 (continue)** | 当前最大可能和 < target → 跳过本轮，继续下一个 i/j | 当前 i/j 太小，但增大 i/j 可能有救 |

> 💡 **break vs continue 的选择逻辑**（排序后通用）：  
> - "**最小**组合都太大" → 后面不会更小 → **break**（彻底放弃当前层及后续）  
> - "**最大**组合都太小" → 当前层换个更大的值可能行 → **continue**（仅跳过当前值）

---

### 复杂度小结

| k | 时间复杂度 |
|---|-----------|
| 2（LC 167）| O(n) |
| 3（LC 15）| O(n²) |
| **4（LC 18）**| **O(n³)** |
| 一般 k | O(n^(k-1)) |

In [ ]:
# ===== 通用模板：kSum（这里展示 4Sum）=====
# 触发：固定数量(≥3)个元素之和=target + 不重复 + 数据量小
# 复杂度：O(n^(k-1)) time | O(1) space（不计输出）

def four_sum_template(nums: List[int], target: int) -> List[List[int]]:
    nums.sort()
    n, res = len(nums), []

    for i in range(n - 3):
        if i > 0 and nums[i] == nums[i - 1]:
            continue
        # 最小和过大 → 后续不可能更小，终止
        if nums[i] + nums[i+1] + nums[i+2] + nums[i+3] > target:
            break
        # 最大和过小 → 当前 i 不可能，跳过
        if nums[i] + nums[n-3] + nums[n-2] + nums[n-1] < target:
            continue

        for j in range(i + 1, n - 2):
            if j > i + 1 and nums[j] == nums[j - 1]:
                continue
            if nums[i] + nums[j] + nums[j+1] + nums[j+2] > target:
                break
            if nums[i] + nums[j] + nums[n-2] + nums[n-1] < target:
                continue

            left, right = j + 1, n - 1
            while left < right:
                total = nums[i] + nums[j] + nums[left] + nums[right]
                if total == target:
                    res.append([nums[i], nums[j], nums[left], nums[right]])
                    while left < right and nums[left] == nums[left + 1]:
                        left += 1
                    while left < right and nums[right] == nums[right - 1]:
                        right -= 1
                    left += 1
                    right -= 1
                elif total < target:
                    left += 1
                else:
                    right -= 1

    return res

print("✅ kSum (4Sum) 模板已加载")

In [ ]:
# ===== LC 18 完整解法 =====
# O(n³) time | O(1) space（不计输出）
# 排序 + 双层循环固定 (i, j) + 对撞双指针 + 四层去重 + 双重剪枝

class Solution_18:
    def fourSum(self, nums: List[int], target: int) -> List[List[int]]:
        quadruplets = list()
        if not nums or len(nums) < 4:
            return quadruplets

        nums.sort()
        length = len(nums)

        for i in range(length - 3):
            if i > 0 and nums[i] == nums[i - 1]:          # 去重①
                continue
            if nums[i] + nums[i+1] + nums[i+2] + nums[i+3] > target:  # 剪枝-下界
                break
            if nums[i] + nums[length-3] + nums[length-2] + nums[length-1] < target:  # 剪枝-上界
                continue

            for j in range(i + 1, length - 2):
                if j > i + 1 and nums[j] == nums[j - 1]:   # 去重②
                    continue
                if nums[i] + nums[j] + nums[j+1] + nums[j+2] > target:
                    break
                if nums[i] + nums[j] + nums[length-2] + nums[length-1] < target:
                    continue

                left, right = j + 1, length - 1
                while left < right:
                    total = nums[i] + nums[j] + nums[left] + nums[right]

                    if total == target:
                        quadruplets.append([nums[i], nums[j], nums[left], nums[right]])
                        while left < right and nums[left] == nums[left + 1]:  # 去重③
                            left += 1
                        while left < right and nums[right] == nums[right - 1]:  # 去重④
                            right -= 1
                        left += 1
                        right -= 1
                    elif total < target:
                        left += 1
                    else:
                        right -= 1

        return quadruplets

print("✅ LC 18 解法已加载")

In [ ]:
# ===== 测试用例 =====

sol18 = Solution_18()

def normalize4(quads):
    return sorted([sorted(q) for q in quads])

tests18 = [
    ([1, 0, -1, 0, -2, 2], 0,  [[-2,-1,1,2],[-2,0,0,2],[-1,0,0,1]]),  # 示例1
    ([2, 2, 2, 2, 2],      8,  [[2,2,2,2]]),                            # 示例2：全相同
    ([0, 0, 0, 0],         0,  [[0,0,0,0]]),                            # 全零
    ([1, 2, 3],            6,  []),                                      # 元素<4个
    ([-3,-2,-1,0,0,1,2,3], 0,  [[-3,-2,2,3],[-3,-1,1,3],[-3,0,0,3],
                                  [-3,0,1,2],[-2,-1,0,3],[-2,-1,1,2],
                                  [-2,0,0,2],[-1,0,0,1]]),               # 复杂多解
]

all_pass = True
for nums, target, expected in tests18:
    result = sol18.fourSum(nums.copy(), target)
    ok = normalize4(result) == normalize4(expected)
    if not ok:
        all_pass = False
    flag = "✅" if ok else "❌"
    print(f"{flag}  nums={nums}, target={target}")
    print(f"      得到 {len(result)} 组, 期望 {len(expected)} 组")
    if not ok:
        print(f"      得到={normalize4(result)}")
        print(f"      期望={normalize4(expected)}")

print()
print("🎉 全部通过！" if all_pass else "⚠️ 有用例未通过")

---
<a id='lc176'></a>
## 3. LC 176 — 第二高的薪水（SQL：DISTINCT + LIMIT OFFSET + NULL处理）

| 字段 | 内容 |
|------|------|
| **题号** | 176 |
| **难度** | 🟢 简单（但 NULL 处理是常见陷阱）|
| **题型** | SQL — 排序去重 + 分页（LIMIT OFFSET）+ 空结果处理 |
| **时间复杂度** | O(n log n)（排序）|
| **空间复杂度** | O(n)（DISTINCT 去重的中间结果）|

### 题目核心
查询 `Employee` 表中**第二高的不同薪水**。  
若不存在第二高薪水（如只有1个不同薪水或表为空），返回 **null**（Pandas 返回 None）。

### ⚡ 触发条件

| 信号 | 操作 |
|------|------|
| "第 N 高 / 第 N 名" | `ORDER BY ... DESC/ASC` + `LIMIT 1 OFFSET (N-1)` |
| 要求**不同**值（去重后排名）| `SELECT DISTINCT` |
| 结果**可能为空**，但仍需返回一行 `NULL` | 用**子查询包裹**（见下）|

---

### 🧠 核心技巧①：LIMIT + OFFSET 实现"第N名"

```sql
SELECT DISTINCT salary
FROM Employee
ORDER BY salary DESC
LIMIT 1 OFFSET 1;     -- 跳过第1名，取接下来的1条 = 第2名
```

> MySQL 简写：`LIMIT 1, 1`（逗号语法：`LIMIT offset, count`，**先 offset 后 count**，容易记反！）  
> 标准 SQL：`LIMIT 1 OFFSET 1`（更不容易出错，推荐）

**"第 N 高"通用公式**：`LIMIT 1 OFFSET (N-1)`，或逗号写法 `LIMIT (N-1), 1`

---

### 🧠 核心技巧②：子查询包裹解决"空结果→NULL"

**问题**：如果直接执行
```sql
SELECT DISTINCT salary FROM Employee ORDER BY salary DESC LIMIT 1 OFFSET 1;
```
当表中只有 1 个不同薪水时，这条语句返回**零行**，而题目要求返回**一行 NULL**。  
"零行"≠"一行NULL" —— 这是 SQL 新手最容易忽略的语义陷阱。

**解决方案**：把上面的查询作为子查询，外层用 `SELECT (...)` 包裹：

```sql
SELECT
    (SELECT DISTINCT salary
     FROM employee
     ORDER BY salary DESC
     LIMIT 1, 1)
AS SecondHighestSalary;
```

**原理**：`SELECT (子查询)` 这种"标量子查询"语法，**即使子查询返回零行，外层 SELECT 仍会返回一行**，  
该列值自动为 `NULL`。这是 SQL 标量子查询的固定行为，恰好满足题目要求。

---

### 对比：两种写法的行为差异

| 写法 | 子查询返回0行时 | 子查询返回1行时 |
|------|----------------|----------------|
| 直接 `SELECT DISTINCT ... LIMIT 1,1` | 返回 **0 行**（结果集为空）❌ | 返回 1 行 |
| `SELECT (SELECT DISTINCT ... LIMIT 1,1) AS x` | 返回 **1 行，值为 NULL** ✅ | 返回 1 行 |

---

### 💡 常见变体 / 关联题

| 变体 | 关键改动 |
|------|---------|
| 第 N 高薪水（LC 177，N 作为参数）| `LIMIT 1 OFFSET (N-1)`，N=0 时需特殊处理返回 NULL |
| 不用 LIMIT，用窗口函数 | `DENSE_RANK() OVER (ORDER BY salary DESC)`，取 rank=2 |
| 每个部门的第二高薪水（LC 184）| 在窗口函数中加 `PARTITION BY department_id` |

```sql
-- 窗口函数写法（更通用，支持"每组第N高"）
SELECT salary AS SecondHighestSalary
FROM (
    SELECT salary,
           DENSE_RANK() OVER (ORDER BY salary DESC) AS rnk
    FROM Employee
) t
WHERE rnk = 2
LIMIT 1;
-- 注意：此写法在"无第二高"时仍返回0行，仍需外层包裹才能得到 NULL
```

In [ ]:
# ===== LC 176 — Pandas 模拟解法 =====
# 对应 SQL: SELECT (SELECT DISTINCT salary FROM employee ORDER BY salary DESC LIMIT 1,1) AS SecondHighestSalary

import pandas as pd
import numpy as np

def second_highest_salary(employee: pd.DataFrame) -> pd.DataFrame:
    distinct_salaries = employee['salary'].drop_duplicates().sort_values(ascending=False)

    if len(distinct_salaries) < 2:
        # 子查询返回0行 → Pandas 中对应返回 None（NULL）
        result = None
    else:
        result = distinct_salaries.iloc[1]   # OFFSET 1 → 第2个（0-indexed 的 index 1）

    return pd.DataFrame({'SecondHighestSalary': [result]})

print("✅ Pandas 解法已加载")

In [ ]:
# ===== 测试用例 =====

# 用例1：示例数据 [100, 200, 300] → 第二高 = 200
df1 = pd.DataFrame({'id': [1, 2, 3], 'salary': [100, 200, 300]})
res1 = second_highest_salary(df1)
print("用例1 (salary=[100,200,300]):")
print(res1.to_string(index=False))
assert res1['SecondHighestSalary'].iloc[0] == 200
print("✅ 通过\n")

# 用例2：只有一个薪水 [100] → 无第二高 → None
df2 = pd.DataFrame({'id': [1], 'salary': [100]})
res2 = second_highest_salary(df2)
print("用例2 (salary=[100]):")
print(res2.to_string(index=False))
assert res2['SecondHighestSalary'].iloc[0] is None
print("✅ 通过（返回 None，而不是空表）\n")

# 用例3：含重复薪水 [100, 100, 200] → DISTINCT后=[200,100] → 第二高=100
df3 = pd.DataFrame({'id': [1, 2, 3], 'salary': [100, 100, 200]})
res3 = second_highest_salary(df3)
print("用例3 (salary=[100,100,200]，含重复):")
print(res3.to_string(index=False))
assert res3['SecondHighestSalary'].iloc[0] == 100
print("✅ 通过（重复值被 DISTINCT 去除，未误判为第二高）")

print()
print("🎉 全部通过！")

---
<a id='summary'></a>
## 4. 模式对照总结

### 三题横向对比

| | LC 209 | LC 18 | LC 176 |
|--|--|--|--|
| **核心题型** | 变长滑动窗口 | 排序+双层循环+对撞双指针 (4Sum) | SQL 分页 + NULL处理 |
| **时间** | O(n) | O(n³) | O(n log n) |
| **空间** | O(1) | O(1) | O(n) |
| **驱动逻辑** | 全正数→单调性→双指针收缩 | kSum降维：固定(k-2)个→两数之和 | LIMIT OFFSET 定位 + 子查询包裹保证非空结果集 |
| **核心决策** | total≥target时用while收缩左边界 | 4层去重 + break(下界)/continue(上界)剪枝 | 标量子查询：0行→自动转为1行NULL |
| **易错点** | 必须 while 不是 if（可能需收缩多次）| break/continue 用反会漏解或低效 | 直接 SELECT 不包裹 → 空结果集≠NULL |

---

### 📐 滑动窗口"求最短" 模板回顾

```python
lk, total, min_len = 0, 0, float('inf')
for rk in range(n):
    total += nums[rk]                  # 扩窗
    while total >= target:             # 满足条件→收缩
        min_len = min(min_len, rk - lk + 1)
        total -= nums[lk]
        lk += 1
return min_len if min_len != float('inf') else 0
```

---

### 📐 kSum 通用递归框架（思维模型）

```
kSum(nums, target, k):
    if k == 2:
        return 对撞双指针找两数之和    # 基本情况：LC 167
    for i in range(n - k + 1):
        if 去重(i): continue
        if 剪枝-下界: break
        if 剪枝-上界: continue
        res += [nums[i]] + kSum(nums[i+1:], target - nums[i], k-1)
    return res

# k=3 → LC 15（一层循环 + 双指针）
# k=4 → LC 18（两层循环 + 双指针）
```

---

### 📐 SQL "第N高 + NULL安全" 万能模板

```sql
SELECT (
    SELECT DISTINCT col
    FROM table_name
    ORDER BY col DESC
    LIMIT 1 OFFSET (N-1)
) AS result_column_name;
```
> 记忆口诀：**"排序去重定位，子查询包裹防空"**

---

### 🗺️ 题型识别速查

```
连续子数组/子串 + 全正数(或非负) + 求最短/最长满足条件   →  变长滑动窗口     (LC 209, LC 3, LC 76)
固定k个元素之和=target (k≥3) + 不含重复元组              →  排序+多层循环+双指针 (LC 15, LC 18)
"第N高/第N名" + 可能不存在时返回NULL                     →  LIMIT OFFSET + 标量子查询包裹 (LC 176, LC 177)
```

---
*复习笔记 · 2026-06 · By Claude*